# 04 — Gold risk score and alerts (rules v1)

Reads `gold_entity_day_features` and writes the daily risk, alert and outcome tables to `lh_gold`.

**Outputs**

| Table | Grain | Purpose |
|---|---|---|
| `gold_risk_daily` | station × date | score, tier, reasons, barrels and USD at risk |
| `gold_alert` | one row per fired alert | fire-on-change log with dedup keys |
| `gold_alert_outcome` | alert × outcome | empty skeleton (preserved if populated) |
| `gold_risk_data_quality` | metric/value | build and suppression checks |

**Rules v1 — deliberately transparent**

- Only **bad-direction** deviations count. `higher_is_bad` comes from the baseline table.
- Each signal is capped at 6 robust-sigma, then weighted:
  `deferment % 0.25 · oil 0.20 · uptime 0.15 · deferred oil 0.10 · target gap 0.10 · THP 0.10 · water cut 0.05 · bean 0.05`.
- `risk_score = 1 - exp(-max(0, weighted - 0.5) / 2)` → 0 to ~0.92.
- **Red** needs multiple corroborating signals, sustained score, or a large value at risk.
- **Amber** also fires on a single signal that stays ≥1.5σ for 3 scored days
  (the "slow deterioration" case) and on material value at risk.
- **Fire on change.** An alert is emitted only when a station moves **into** Amber/Red or escalates
  to Red. Same-tier days are suppressed; a cooldown stops flip-flopping.
- **No reasons, no alert.** Every alert carries its top bad-direction reasons.
- Estimation-window rows are `Unscored` and can never alert. Unobserved days are `NoData`.

> This is the honest MVP. Layer 2 (a learned onset model) replaces the weights only if it beats
> these rules on lead time at the same alert budget.


In [ ]:
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType, DateType, DoubleType, LongType, StringType, StructField, StructType,
    TimestampType,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

# --- sandbox identifiers ---------------------------------------------------
WORKSPACE_ID = "355cd573-8b17-430a-b606-c6b9424d36a1"
GOLD_ID = "df9a08e5-b370-4c2a-b608-6d0f326003ef"

GOLD = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{GOLD_ID}"

BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
COMPUTED_AT = datetime.now(timezone.utc).replace(tzinfo=None)

RULESET_VERSION = "risk-rules-v1"

# --- rules v1 configuration ------------------------------------------------
METRICS = [
    # name, source column, higher_is_bad, label, kind
    ("deferment_pct",  "deferment_pct",   True,  "deferment %",  "ratio"),
    ("oil",            "actual_oil",      False, "oil",          "level"),
    ("uptime",         "uptime_avg",      False, "uptime",       "level"),
    ("deferred_oil",   "deferred_oil",    True,  "deferred oil", "level"),
    ("target_gap_pct", "target_gap_pct",  True,  "gap to target", "ratio"),
    ("thp",            "thp_avg",         False, "THP",          "level"),
    ("water_cut",      "water_rf_avg",    True,  "water cut",    "ratio"),
    ("bean",           "bean_avg",        False, "bean/choke",   "level"),
]
METRIC_META = {
    m[0]: {"column": m[1], "higher_is_bad": m[2], "label": m[3], "kind": m[4]}
    for m in METRICS
}
WEIGHTS = {
    "deferment_pct": 0.25,
    "oil": 0.20,
    "uptime": 0.15,
    "deferred_oil": 0.10,
    "target_gap_pct": 0.10,
    "thp": 0.10,
    "water_cut": 0.05,
    "bean": 0.05,
}
BAD_Z_CAP = 6.0
RISK_FLOOR = 0.5
SCORE_SCALE = 2.0

PRIMARY_SIGNALS = ["oil", "deferment_pct", "deferred_oil", "thp"]
PRIMARY_SIGNAL_MIN_Z = 1.5

AMBER_SCORE = 0.40
AMBER_PERSIST_SCORE = 0.30
AMBER_VALUE_USD = 50_000
AMBER_SIGNAL_Z = 1.4
AMBER_SIGNAL_RUN = 3

RED_SCORE = 0.70
RED_PERSIST_SCORE = 0.55
RED_VALUE_USD = 250_000

COOLDOWN_DAYS = 3


def read_gold(name: str):
    return spark.read.format("delta").load(f"{GOLD}/Tables/{name}")


def write_gold(name: str, sdf) -> None:
    path = f"{GOLD}/Tables/{name}"
    (sdf.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(path))
    spark.sql(f"CREATE TABLE IF NOT EXISTS {name} USING DELTA LOCATION '{path}'")
    print(f"{name}: {sdf.count():,} rows")


print(f"Batch: {BATCH_ID}  ruleset: {RULESET_VERSION}")

In [ ]:
# --- read gold features + the deferment baseline --------------------------
cols = [
    "station_id", "station_name", "asset", "owner_name", "owner_upn",
    "date", "weekday", "is_weekend", "is_observed", "baseline_phase",
    "baseline_version",
    "actual_oil", "deferred_oil", "potential_oil", "deferment_pct",
    "target_oil", "target_gap_pct", "uptime_avg", "thp_avg", "bean_avg",
    "water_rf_avg",
    "deferment_pct_roll_mean_7", "deferred_oil_roll_mean_7",
    "deferred_value_usd_roll_sum_7", "event_count_roll_sum_7",
    "oil_price_usd", "deferred_value_usd", "potential_value_usd",
    "top_signal", "top_signal_z",
]
for m in METRIC_META:
    cols += [f"z_{m}", f"dev_{m}", f"expected_{m}"]

features = read_gold("gold_entity_day_features").select(*cols)
risk = features.toPandas()
risk["date"] = pd.to_datetime(risk["date"])
risk = risk.sort_values(["station_id", "date"]).reset_index(drop=True)

base = (
    read_gold("gold_entity_baseline")
    .filter(F.col("metric") == "deferment_pct")
    .select(
        F.col("entity_id").alias("station_id"),
        F.col("median").alias("baseline_defpct"),
        F.col("p90").alias("baseline_defpct_p90"),
    )
    .toPandas()
)
risk = risk.merge(base, on="station_id", how="left")
risk["baseline_defpct"] = risk["baseline_defpct"].fillna(0.0)
risk["baseline_defpct_p90"] = risk["baseline_defpct_p90"].fillna(risk["baseline_defpct"])

print(f"feature rows: {len(risk):,}")
display(
    risk.groupby("baseline_phase")
    .agg(rows=("date", "size"), observed=("is_observed", "sum"))
)

In [ ]:
# --- 1. bad-direction deviation scores ------------------------------------
eligible = risk["is_observed"] & (risk["baseline_phase"] == "scored")

bad_matrix = {}
for m, meta in METRIC_META.items():
    z = risk[f"z_{m}"].fillna(0.0)
    bad = np.maximum(0.0, z) if meta["higher_is_bad"] else np.maximum(0.0, -z)
    bad_matrix[m] = np.minimum(bad, BAD_Z_CAP)

weighted = sum(WEIGHTS[m] * bad_matrix[m] for m in METRIC_META)
max_bad = np.maximum.reduce([bad_matrix[m] for m in METRIC_META])

primary_bad = np.maximum.reduce([bad_matrix[m] for m in PRIMARY_SIGNALS])
risk["max_bad_z"] = np.where(eligible, max_bad, np.nan)
risk["primary_bad_z"] = np.where(eligible, primary_bad, np.nan)
risk["weighted_severity"] = np.where(eligible, weighted, np.nan)
raw_score = 1.0 - np.exp(-np.maximum(0.0, weighted - RISK_FLOOR) / SCORE_SCALE)
risk["risk_score"] = np.where(eligible, raw_score, np.nan)
risk["risk_signals"] = np.where(
    eligible,
    sum((bad_matrix[m] >= 1.0).astype(int) for m in METRIC_META),
    0,
)

# --- 2. persistence of a deteriorating signal -----------------------------
# Consecutive eligible days. An unobserved day breaks the run (we do not know).
persistence_days = []
signal_run = []
current_score_run = {}
current_signal_run = {}
for _, row in risk.iterrows():
    sid = row["station_id"]
    if not row["is_observed"] or row["baseline_phase"] != "scored":
        current_score_run[sid] = 0
        current_signal_run[sid] = 0
        persistence_days.append(0)
        signal_run.append(0)
        continue
    current_score_run[sid] = current_score_run.get(sid, 0) + 1 if row["risk_score"] >= 0.35 else 0
    current_signal_run[sid] = current_signal_run.get(sid, 0) + 1 if row["max_bad_z"] >= AMBER_SIGNAL_Z else 0
    persistence_days.append(current_score_run[sid])
    signal_run.append(current_signal_run[sid])
risk["persistence_days"] = persistence_days
risk["signal_run"] = signal_run

# --- 3. value at risk ------------------------------------------------------
# Expected deferred rate moves from the station's normal towards its own p90 as
# severity rises, and never below today's / last week's observed rate.
severity = np.clip(
    (risk["weighted_severity"].fillna(0.0) - RISK_FLOOR) / (BAD_Z_CAP - RISK_FLOOR),
    0.0, 1.0,
)
expected_rate = risk["baseline_defpct"] + (
    np.maximum(risk["baseline_defpct_p90"], risk["baseline_defpct"]) - risk["baseline_defpct"]
) * severity
expected_rate = np.maximum.reduce([
    expected_rate.to_numpy(),
    risk["deferment_pct"].fillna(0.0).to_numpy(),
    risk["deferment_pct_roll_mean_7"].fillna(0.0).to_numpy(),
])
# NB: `expected_deferment_pct` is the baseline normal from nb_03 and is kept
# for explainability. The risk-adjusted rate gets its own name so we never
# overwrite the value used in the reason text.
risk["predicted_deferment_pct"] = np.clip(expected_rate, 0.0, 1.0)
risk["exp_deferred_bbl"] = risk["potential_oil"] * risk["predicted_deferment_pct"]
risk["exp_excess_bbl"] = risk["potential_oil"] * np.maximum(
    0.0, risk["predicted_deferment_pct"] - risk["baseline_defpct"]
)
risk["exp_usd"] = risk["exp_excess_bbl"] * risk["oil_price_usd"]
risk.loc[~risk["is_observed"], ["predicted_deferment_pct", "exp_deferred_bbl", "exp_excess_bbl", "exp_usd"]] = np.nan

# --- 4. explainable, bad-direction-only reasons ---------------------------
RATIO_METRICS = {"deferment_pct", "water_cut", "target_gap_pct"}


def reason_text(row, m: str) -> str:
    z = row[f"z_{m}"]
    if pd.isna(z):
        return ""
    meta = METRIC_META[m]
    if (z > 0) != meta["higher_is_bad"] or z == 0:
        return ""
    value = row[meta["column"]]
    expected = row[f"expected_{m}"]
    change = ""
    if pd.notna(value) and pd.notna(expected):
        delta = abs(value - expected)
        if m in RATIO_METRICS and delta >= 0.0005:
            change = f"{delta * 100:.0f}pp "
        elif abs(expected) > 1e-9:
            pct = delta / abs(expected) * 100
            change = f"{pct:.0f}% " if pct < 1000 else ">999% "
    direction = "above" if z > 0 else "below"
    return f"{meta['label']} {change}{direction} its normal (z={z:+.1f})"


def reasons_for_row(row):
    items = []
    for m in METRIC_META:
        z = row[f"z_{m}"]
        if pd.isna(z):
            continue
        meta = METRIC_META[m]
        is_bad = (z > 0) == meta["higher_is_bad"] and z != 0
        if not is_bad or abs(z) < 1.0:
            continue
        text = reason_text(row, m)
        if text:
            items.append({
                "metric": m,
                "label": meta["label"],
                "z": round(float(z), 2),
                "direction": "above" if z > 0 else "below",
                "value": None if pd.isna(row[meta["column"]]) else float(row[meta["column"]]),
                "expected": None if pd.isna(row[f"expected_{m}"]) else float(row[f"expected_{m}"]),
                "text": text,
            })
    items.sort(key=lambda x: abs(x["z"]), reverse=True)
    return items


reasons_json = []
reasons_text = []
top_risk_signal = []
top_risk_z = []
for _, row in risk.iterrows():
    items = reasons_for_row(row) if row["is_observed"] and row["baseline_phase"] == "scored" else []
    reasons_json.append(json.dumps(items[:5]))
    reasons_text.append("; ".join(i["text"] for i in items[:3]))
    if items:
        top_risk_signal.append(items[0]["metric"])
        top_risk_z.append(items[0]["z"])
    else:
        top_risk_signal.append("")
        top_risk_z.append(np.nan)

risk["risk_reasons_json"] = reasons_json
risk["risk_reasons"] = reasons_text
risk["top_risk_signal"] = top_risk_signal
risk["top_risk_z"] = top_risk_z
risk["risk_direction"] = np.select(
    [~eligible, risk["max_bad_z"] >= 1.0, risk["max_bad_z"].isna()],
    ["not_scored", "deteriorating", "unknown"],
    default="stable",
)

display(
    risk.loc[eligible]
    .sort_values(["risk_score", "exp_usd"], ascending=False)
    .loc[:, ["station_name", "date", "risk_score", "max_bad_z", "risk_signals",
             "exp_deferred_bbl", "exp_usd", "top_risk_signal", "risk_reasons"]]
    .head(10)
)

In [ ]:
# --- 5. tiers --------------------------------------------------------------
def assign_tier(row) -> str:
    if not row["is_observed"]:
        return "NoData"
    if row["baseline_phase"] != "scored":
        return "Unscored"

    score = row["risk_score"]
    if pd.isna(score):
        return "NoData"
    exp_usd = 0.0 if pd.isna(row["exp_usd"]) else row["exp_usd"]
    mbz = 0.0 if pd.isna(row["max_bad_z"]) else row["max_bad_z"]

    if (
        score >= RED_SCORE
        or (score >= RED_PERSIST_SCORE and row["persistence_days"] >= 2)
        or (mbz >= 6 and row["signal_run"] >= 3 and score >= 0.45)
        or (exp_usd >= RED_VALUE_USD and score >= 0.30)
    ):
        return "Red"
    if (
        score >= AMBER_SCORE
        or (score >= AMBER_PERSIST_SCORE and row["persistence_days"] >= 2)
        or (
            mbz >= AMBER_SIGNAL_Z
            and row["signal_run"] >= AMBER_SIGNAL_RUN
            and row["primary_bad_z"] >= PRIMARY_SIGNAL_MIN_Z
        )
        or (exp_usd >= AMBER_VALUE_USD and score >= 0.20)
    ):
        return "Amber"
    return "Green"


risk["risk_tier"] = risk.apply(assign_tier, axis=1)

# --- 6. fire-on-change alerts ---------------------------------------------
TIER_RANK = {"Green": 0, "Amber": 1, "Red": 2}
alerts = []
alert_action = []
suppression_reason = []

prev_tier = {}
last_alert_date = {}
last_alert_tier = {}

for _, row in risk.iterrows():
    sid = row["station_id"]
    tier = row["risk_tier"]
    if tier not in TIER_RANK:
        alert_action.append("None")
        suppression_reason.append("not_eligible")
        continue

    current_rank = TIER_RANK[tier]
    previous = prev_tier.get(sid, "Green")
    previous_rank = TIER_RANK[previous]
    action, suppress = "None", ""

    if current_rank > previous_rank:
        fire = True
        if sid in last_alert_date:
            days_since = (row["date"] - last_alert_date[sid]).days
            escalation = tier == "Red" and last_alert_tier.get(sid) == "Amber"
            if days_since < COOLDOWN_DAYS and not escalation:
                fire = False
                suppress = "cooldown"
        if fire:
            alert_date = row["date"].date()
            alert_key = f"{sid}|{alert_date}|{tier}|{RULESET_VERSION}"
            alert_id = hashlib.md5(alert_key.encode("utf-8")).hexdigest()[:16]
            alert_type = (
                "escalated_to_red" if previous == "Amber"
                else ("entered_red" if tier == "Red" else "entered_amber")
            )
            alerts.append({
                "alert_id": alert_id,
                "alert_key": alert_key,
                "station_id": sid,
                "station_name": row["station_name"],
                "asset": row["asset"],
                "owner_name": row["owner_name"],
                "owner_upn": row["owner_upn"],
                "alert_date": alert_date,
                "tier": tier,
                "previous_tier": previous,
                "alert_type": alert_type,
                "risk_score": float(row["risk_score"]),
                "risk_reasons": row["risk_reasons"],
                "risk_reasons_json": row["risk_reasons_json"],
                "top_signal": row["top_risk_signal"],
                "top_signal_z": None if pd.isna(row["top_risk_z"]) else float(row["top_risk_z"]),
                "persistence_days": int(row["persistence_days"]),
                "signal_run": int(row["signal_run"]),
                "exp_deferred_bbl": None if pd.isna(row["exp_deferred_bbl"]) else float(row["exp_deferred_bbl"]),
                "exp_usd": None if pd.isna(row["exp_usd"]) else float(row["exp_usd"]),
                "ruleset_version": RULESET_VERSION,
                "baseline_version": row["baseline_version"],
                "fired_at_utc": COMPUTED_AT,
                "alert_status": "New",
                "notification_channel": "Teams",
                "notification_target": row["owner_upn"],
            })
            action = "Fired"
            last_alert_date[sid] = row["date"]
            last_alert_tier[sid] = tier
        else:
            action = "Suppressed"
    elif current_rank == previous_rank:
        if tier in ("Amber", "Red"):
            action, suppress = "Suppressed", "no_change"
    else:
        action, suppress = "Suppressed", "de_escalation"

    alert_action.append(action)
    suppression_reason.append(suppress)
    prev_tier[sid] = tier

risk["alert_action"] = alert_action
risk["suppression_reason"] = suppression_reason
risk["alert_eligible"] = risk["risk_tier"].isin(["Amber", "Red"])
risk["is_latest_date"] = risk["date"] == risk["date"].max()

alerts_df = pd.DataFrame(alerts)
if len(alerts_df):
    alerts_df = alerts_df.sort_values(["alert_date", "tier", "exp_usd"], ascending=[True, True, False])

print("tier counts (scored, observed):")
display(risk.loc[eligible, "risk_tier"].value_counts())
print(f"alerts fired: {len(alerts_df)}")
if len(alerts_df):
    display(alerts_df.loc[:, ["alert_date", "station_name", "tier", "previous_tier",
                             "alert_type", "risk_score", "exp_usd", "top_signal"]])

assert len(risk) == risk[["station_id", "date"]].drop_duplicates().shape[0]
assert not risk.loc[~eligible, "alert_eligible"].any()
if len(alerts_df):
    assert alerts_df["alert_key"].is_unique
print("alert invariants: OK")

In [ ]:
# --- 7. write tables -------------------------------------------------------
risk_out = risk.copy()
risk_out["date"] = risk_out["date"].dt.date

risk_cols = [
    "station_id", "station_name", "asset", "owner_name", "owner_upn",
    "date", "weekday", "is_weekend", "is_observed", "baseline_phase",
    "risk_score", "risk_tier", "risk_signals", "risk_direction",
    "persistence_days", "signal_run", "max_bad_z", "primary_bad_z", "weighted_severity",
    "risk_reasons", "risk_reasons_json", "top_risk_signal", "top_risk_z",
    "expected_deferment_pct", "predicted_deferment_pct",
    "exp_deferred_bbl", "exp_excess_bbl", "exp_usd",
    "potential_oil", "deferred_oil", "deferment_pct", "oil_price_usd",
    "deferred_value_usd", "deferred_value_usd_roll_sum_7", "event_count_roll_sum_7",
    "alert_eligible", "alert_action", "suppression_reason", "is_latest_date",
    "ruleset_version", "baseline_version", "computed_at_utc",
]
risk_out["ruleset_version"] = RULESET_VERSION
risk_out["computed_at_utc"] = COMPUTED_AT
write_gold("gold_risk_daily", spark.createDataFrame(risk_out.loc[:, risk_cols]))

if len(alerts_df):
    write_gold("gold_alert", spark.createDataFrame(alerts_df))
else:
    # Still create the table with a schema so the report has something to bind.
    empty_alert = spark.createDataFrame([], StructType([
        StructField("alert_id", StringType()), StructField("alert_key", StringType()),
        StructField("station_id", LongType()), StructField("station_name", StringType()),
        StructField("asset", StringType()), StructField("owner_name", StringType()),
        StructField("owner_upn", StringType()), StructField("alert_date", DateType()),
        StructField("tier", StringType()), StructField("previous_tier", StringType()),
        StructField("alert_type", StringType()), StructField("risk_score", DoubleType()),
        StructField("risk_reasons", StringType()), StructField("risk_reasons_json", StringType()),
        StructField("top_signal", StringType()), StructField("top_signal_z", DoubleType()),
        StructField("persistence_days", LongType()), StructField("signal_run", LongType()),
        StructField("exp_deferred_bbl", DoubleType()), StructField("exp_usd", DoubleType()),
        StructField("ruleset_version", StringType()), StructField("baseline_version", StringType()),
        StructField("fired_at_utc", TimestampType()), StructField("alert_status", StringType()),
        StructField("notification_channel", StringType()), StructField("notification_target", StringType()),
    ]))
    write_gold("gold_alert", empty_alert)

# Never wipe outcomes captured by the human feedback loop.
if spark.catalog.tableExists("gold_alert_outcome"):
    print("gold_alert_outcome already exists - preserving recorded outcomes.")
else:
    outcome_schema = StructType([
        StructField("alert_id", StringType()),
        StructField("outcome_recorded_at_utc", TimestampType()),
        StructField("event_occurred", BooleanType()),
        StructField("event_date", DateType()),
        StructField("actual_lead_days", LongType()),
        StructField("detection_type", StringType()),
        StructField("disposition", StringType()),
        StructField("acknowledged_by", StringType()),
        StructField("acknowledged_at_utc", TimestampType()),
        StructField("action_note", StringType()),
        StructField("false_reason", StringType()),
        StructField("updated_at_utc", TimestampType()),
    ])
    write_gold("gold_alert_outcome", spark.createDataFrame([], outcome_schema))

In [ ]:
# --- 8. data quality + display --------------------------------------------
scored = risk.loc[eligible]
tier_counts = scored["risk_tier"].value_counts().to_dict()
alerts_n = len(alerts_df)
top_alert = "none"
if alerts_n:
    top_row = alerts_df.sort_values("exp_usd", ascending=False).iloc[0]
    top_alert = (f"{top_row['station_name']} {top_row['alert_date']} {top_row['tier']} "
                 f"${top_row['exp_usd']:,.0f}")

dq = [
    ("batch_id", BATCH_ID),
    ("ruleset_version", RULESET_VERSION),
    ("risk_rows", len(risk)),
    ("scored_observed_rows", len(scored)),
    ("estimation_rows", int(((risk["baseline_phase"] == "estimation") & risk["is_observed"]).sum())),
    ("nodata_rows", int((~risk["is_observed"]).sum())),
    ("red_days", tier_counts.get("Red", 0)),
    ("amber_days", tier_counts.get("Amber", 0)),
    ("green_days", tier_counts.get("Green", 0)),
    ("alerts_fired", alerts_n),
    ("alerts_red", 0 if not alerts_n else int((alerts_df["tier"] == "Red").sum())),
    ("alerts_amber", 0 if not alerts_n else int((alerts_df["tier"] == "Amber").sum())),
    ("suppressed_no_change", int((risk["suppression_reason"] == "no_change").sum())),
    ("suppressed_cooldown", int((risk["suppression_reason"] == "cooldown").sum())),
    ("suppressed_de_escalation", int((risk["suppression_reason"] == "de_escalation").sum())),
    ("max_risk_score", round(float(risk["risk_score"].max()), 4)),
    ("latest_date", str(risk["date"].max().date())),
    ("latest_alerts", 0 if not alerts_n else int((alerts_df["alert_date"] == risk["date"].max().date()).sum())),
    ("top_alert", top_alert),
    ("min_date", str(risk["date"].min().date())),
]
dq_df = pd.DataFrame(dq, columns=["metric", "value"])
dq_df["value"] = dq_df["value"].astype(str)
write_gold("gold_risk_data_quality", spark.createDataFrame(dq_df))

print("\nRisk build complete.")
display(spark.sql("SHOW TABLES").filter(F.col("tableName").startswith("gold_")).orderBy("tableName"))
display(dq_df)

## Next

1. **Semantic model + Power BI report** over `lh_gold` (Direct Lake): risk ranking,
   station drill-down, deviation heatmap, alert log and outcome view.
2. **Orchestration** of `01 → 02 → 03 → 04` in a Data Factory pipeline, daily.
3. **Outcome job** that fills `gold_alert_outcome` by checking whether a coded
   onset or large-loss day followed each alert, then computes realised lead time.
4. **Layer 2 onset model** only if it beats these rules on lead time at the same
   alert budget.

> In production, merge `gold_alert` by `alert_key` instead of overwriting it, so
> delivery/acknowledgement state survives a rerun. This sandbox run is a
> deterministic backfill; `gold_alert_outcome` is preserved either way.
